# 4.7 — Data Validation and Profiling

**Chapter 4, section 4.5 "Checking That the Data Is What It Is Assumed to Be"**, and the
starting point for **Exercise 7**.

**The question this notebook answers:** every technique in this chapter acts on beliefs about
the data — that fares are numbers, that timestamps parse, that medallions identify vehicles,
that yesterday's row count resembles today's. Most pipeline failures are not exceptions; they
are beliefs that quietly stopped being true.

Two steps make the beliefs themselves an object of engineering:

1. **Profiling** — discovering what an unfamiliar dataset actually contains, before any belief
   is formed, in a single pass.
2. **Validation** — turning the beliefs a pipeline depends on into machine-checked expectations
   that run with the pipeline, at a cost it will tolerate forever.

**Data.** The taxi file, and **a deliberately corrupted copy of it** written to scratch, with
four defects that the checks below are supposed to catch. A validation suite that has never
failed is not a validation suite; it is an assertion about the author's luck.

Runs on a laptop in about a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, time, logging
from pyspark.sql import SparkSession, Observation
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-4.7")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)
spark.conf.set("spark.sql.session.timeZone", "UTC")

names = ["medallion", "hack_license", "pickup_datetime", "dropoff_datetime",
         "trip_time", "trip_distance", "pickup_longitude", "pickup_latitude",
         "dropoff_longitude", "dropoff_latitude", "payment_type", "fare_amount",
         "surcharge", "mta_tax", "tip_amount", "tolls_amount", "total_amount"]
types = {"medallion": StringType(), "hack_license": StringType(),
         "pickup_datetime": TimestampType(), "dropoff_datetime": TimestampType(),
         "trip_time": IntegerType(), "payment_type": StringType()}
schema = StructType([StructField(n, types.get(n, DoubleType()), True) for n in names])

df = (spark.read.schema(schema).option("header", "false")
      .csv(f"{DATA}/taxi-data-sorted-small.csv.bz2")
      .drop("pickup_longitude", "pickup_latitude",
            "dropoff_longitude", "dropoff_latitude")
      .cache())
print("Spark", spark.version)

Spark 4.2.0


## 1. Profiling an unfamiliar dataset

### First contact

The course notebooks already model the standard first three steps. The fourth call is the one
that needs a warning attached.

In [2]:
print(f"rows: {df.count():,}")
df.show(5, truncate=False)
df.printSchema()

rows: 1,999,999
+--------------------------------+--------------------------------+-------------------+-------------------+---------+-------------+------------+-----------+---------+-------+----------+------------+------------+
|medallion                       |hack_license                    |pickup_datetime    |dropoff_datetime   |trip_time|trip_distance|payment_type|fare_amount|surcharge|mta_tax|tip_amount|tolls_amount|total_amount|
+--------------------------------+--------------------------------+-------------------+-------------------+---------+-------------+------------+-----------+---------+-------+----------+------------+------------+
|07290D3599E7A0D62097A346EFCC1FB5|E7750A37CAB07D0DFF0AF7E3573AC141|2013-01-01 00:00:00|2013-01-01 00:02:00|120      |0.44         |CSH         |3.5        |0.5      |0.5    |0.0       |0.0         |4.5         |
|22D70BF00EEB0ADC83BA8177BB861991|3FF2709163DE7036FCAA4E5A3324E4BF|2013-01-01 00:02:00|2013-01-01 00:02:00|0        |0.0          |CSH  

In [3]:
# toPandas() collects the ENTIRE distributed dataset into the driver's memory. It is safe
# here only because the result has already been reduced to five rows -- which is the rule,
# not the exception.
by_payment = (df.groupBy("payment_type").agg(F.count("*").alias("trips"))
              .orderBy(F.desc("trips")).toPandas())
print(by_payment.to_string(index=False))
print(f"\n{len(by_payment)} rows collected. The same call on `df` would move "
      f"{df.count():,} rows onto one machine.")

payment_type   trips
         CRD 1011236
         CSH  987134
         UNK    1626
         DIS       2
         NOC       1

5 rows collected. The same call on `df` would move 1,999,999 rows onto one machine.


### `summary()` earns its upgrade over `describe()` on one column

`describe()` reports count, mean, standard deviation, minimum and maximum. `summary()` adds the
approximate quartiles, and the quartiles are what make the difference.

In [4]:
df.select("fare_amount").summary().show()

+-------+------------------+
|summary|       fare_amount|
+-------+------------------+
|  count|           1999999|
|   mean|11.898484839242423|
| stddev|10.173323445098891|
|    min|               2.5|
|    25%|               6.5|
|    50%|               9.0|
|    75%|              13.0|
|    max|             465.0|
+-------+------------------+



In [5]:
row = df.select(
    F.round(F.avg("fare_amount"), 2).alias("mean"),
    F.expr("percentile_approx(fare_amount, 0.5)").alias("median"),
    F.max("fare_amount").alias("max"),
    F.round(F.expr("percentile_approx(fare_amount, 0.99)"), 2).alias("p99")).first()
print(f"mean ${row['mean']}   median ${row['median']}   99th percentile ${row['p99']}"
      f"   maximum ${row['max']}")
print(f"\nthe maximum is {row['max'] / row['median']:.0f} times the median, and "
      f"{row['max'] / row['p99']:.0f} times the 99th percentile.")

mean $11.9   median $9.0   99th percentile $52.0   maximum $465.0

the maximum is 52 times the median, and 9 times the 99th percentile.


A median near \$9 beside a mean near \$12 and a maximum two orders of magnitude above both: the
column has a long right tail, which will dominate sums and averages, distort a mean-based
imputation, and deserve inspection before any of the three. **A mean is an answer; a median
sitting far from it is a warning.**

### The whole profile in a single pass

The first two profiling questions — how much is missing, how many distinct values are there —
are answered for every column at once, with the aggregation assembled by comprehensions on the
driver.

In [6]:
numeric = ["trip_time", "trip_distance", "fare_amount", "total_amount"]

profile = df.agg(
    F.count("*").alias("rows"),
    *[F.count(F.when(F.col(c).isNull(), c)).alias(f"{c}_nulls") for c in numeric],
    *[F.approx_count_distinct(c).alias(f"{c}_distinct") for c in numeric])

profile.show(vertical=True, truncate=False)
print("Exchange nodes in that plan:",
      profile._jdf.queryExecution().executedPlan().toString().count("Exchange"),
      " -- nine metrics, one pass over the data")

-RECORD 0-------------------------
 rows                   | 1999999 
 trip_time_nulls        | 0       
 trip_distance_nulls    | 0       
 fare_amount_nulls      | 0       
 total_amount_nulls     | 0       
 trip_time_distinct     | 1886    
 trip_distance_distinct | 3292    
 fare_amount_distinct   | 568     
 total_amount_distinct  | 4040    

Exchange nodes in that plan: 1  -- nine metrics, one pass over the data


Two mechanisms make that listing work. `F.count` over an expression counts only the rows where
the expression is non-null, so `F.count(F.when(condition, c))` is *the* idiom for counting rows
that satisfy a condition. And the list comprehensions run on the driver at planning time,
assembling one wide aggregation from a column list — the same expression-building pattern as the
array rewrite in [4.5](04.05%20Array%20Functions%20without%20UDFs.ipynb).

### Reading cardinalities against the row count

Four readings recur, and all four are present in this file.

In [7]:
rows = df.count()

# The wide keys use the sketch rather than an exact count, deliberately: an exact distinct
# over a thirteen-column key has to remember every distinct row it has seen. Asking for it
# on this file is how you meet `java.lang.OutOfMemoryError`, which is the lesson notebook
# 4.4 states about exact distinctness and collect_list, arriving here uninvited.
candidates = {
    "payment_type":                F.count_distinct("payment_type"),            # 5 values
    "medallion":                   F.approx_count_distinct("medallion"),
    "medallion + pickup_datetime": F.approx_count_distinct(
                                       F.hash("medallion", "pickup_datetime")),
    "the whole row":               F.approx_count_distinct(F.hash(*df.columns)),
}
counted = df.agg(*[expression.alias(name.replace(" ", "_").replace("+", "and"))
                   for name, expression in candidates.items()]).first()

print(f"{'column(s)':30s} {'distinct':>10s} {'of rows':>10s}  reading")
for name in candidates:
    n = counted[name.replace(" ", "_").replace("+", "and")]
    share = n / rows
    if share < 0.001:
        reading = "categorical: a label, whatever its name says"
    elif share < 0.5:
        reading = f"a repeated identifier: ~{rows / n:,.0f} rows per value"
    elif share >= 0.99:
        reading = "a candidate key (within the sketch's error)"
    else:
        reading = "SHOULD be a key and is not -> duplicates"
    print(f"{name:30s} {n:>10,} {rows:>10,}  {reading}")

column(s)                        distinct    of rows  reading
payment_type                            5  1,999,999  categorical: a label, whatever its name says
medallion                          10,682  1,999,999  a repeated identifier: ~187 rows per value
medallion + pickup_datetime     1,950,354  1,999,999  SHOULD be a key and is not -> duplicates
the whole row                   2,099,787  1,999,999  a candidate key (within the sketch's error)


The third line is the valuable one. A trip is identified by its cab and its pickup time, so
`medallion + pickup_datetime` *ought* to be unique — and it is not. The shortfall is the number
of trips that share a cab and a minute, which is a duplicate problem discovered here, by
counting, rather than downstream after a join has multiplied it. (Whether those rows are
genuine duplicates or two legitimate records at the same minute is not a question counting can
settle; what counting settles is that the column pair cannot be used as a key.)

Note the instrument, too. Three of the four counts are sketches rather than exact counts,
because an exact distinct over a wide key must remember every distinct value it sees: asking
for `count_distinct` over all thirteen columns of this file raises
`java.lang.OutOfMemoryError` on a laptop. Hashing the columns first and estimating is the
version that runs, and the estimate is good to a few percent.

## 2. Expressing expectations as checks

Profiling is exploratory and manual. Validation is its industrialized form: the facts the
pipeline depends on, written down as checks that run every time it runs. Three kinds cover the
standard failure surface, and each catches what the others cannot.

### A corrupted batch, so the checks have something to catch

Four defects, each aimed at a different check.

In [8]:
BAD = os.path.join(SCRATCH, "ch04-bad-batch")
GOOD = os.path.join(SCRATCH, "ch04-good-batch")

if not os.path.exists(GOOD):
    df.write.mode("overwrite").parquet(GOOD)

if not os.path.exists(BAD):
    bucket = F.pmod(F.hash("medallion", "pickup_datetime"), F.lit(100))
    (df
     # 1. a half-delivered feed
     .where(bucket < 50)
     # 2. negative fares: a value of the right type that cannot be true
     .withColumn("fare_amount",
                 F.when(bucket == 3, -F.col("fare_amount")).otherwise(F.col("fare_amount")))
     # 3. missing fares where none were missing before
     .withColumn("fare_amount",
                 F.when(bucket == 7, None).otherwise(F.col("fare_amount")))
     # 4. the column shift of 4.1, at DataFrame level: every money column one place left,
     #    which leaves the types intact and the meanings wrong
     .withColumn("shifted", bucket == 11)
     .withColumn("surcharge",    F.when(F.col("shifted"), F.col("mta_tax")).otherwise(F.col("surcharge")))
     .withColumn("mta_tax",      F.when(F.col("shifted"), F.col("tip_amount")).otherwise(F.col("mta_tax")))
     .withColumn("tip_amount",   F.when(F.col("shifted"), F.col("tolls_amount")).otherwise(F.col("tip_amount")))
     .withColumn("tolls_amount", F.when(F.col("shifted"), F.col("total_amount")).otherwise(F.col("tolls_amount")))
     .withColumn("total_amount", F.when(F.col("shifted"), F.lit(0.0)).otherwise(F.col("total_amount")))
     .drop("shifted")
     .write.mode("overwrite").parquet(BAD))

good = spark.read.parquet(GOOD)
bad = spark.read.parquet(BAD)
print(f"good batch: {good.count():,} rows")
print(f"bad batch : {bad.count():,} rows")

good batch: 1,999,999 rows
bad batch : 998,885 rows


### Check 1: the schema, which is free

`df.dtypes` reads the schema. It scans no data at all, so this check costs nothing and catches
what nothing else can: an upstream rename or a type change.

In [9]:
expected = {"pickup_datetime": "timestamp", "fare_amount": "double",
            "medallion": "string", "trip_time": "int"}

def schema_check(dataframe, label):
    actual = dict(dataframe.dtypes)               # reads the schema; scans no data
    if (drift := {c: (t, actual.get(c)) for c, t in expected.items() if actual.get(c) != t}):
        print(f"  {label:22s} FAIL  schema drift: {drift}")
        return False
    print(f"  {label:22s} pass")
    return True

for label, batch in [("the good batch", good), ("the bad batch", bad)]:
    schema_check(batch, label)

# A drifted batch: the source starts sending the fare as text, and renames a column.
drifted = (bad.withColumn("fare_amount", F.col("fare_amount").cast("string"))
              .withColumnRenamed("medallion", "cab_id"))
schema_check(drifted, "a drifted batch")
print("\nthe drift report names, per column, what was expected and what arrived;")
print("`None` means the column is not there at all under that name.")

  the good batch         pass
  the bad batch          pass
  a drifted batch        FAIL  schema drift: {'fare_amount': ('double', 'string'), 'medallion': ('string', None)}

the drift report names, per column, what was expected and what arrived;
`None` means the column is not there at all under that name.


The bad batch **passes** the schema check, and that is the honest result: its defects are
values, not types. The comparison uses Python's assignment expression (`:=`, which assigns a
value and tests it in one step) to keep the check to three lines.

### Checks 2 and 3: ranges and row counts, in one shared scan

A range check catches values of the correct type that are nonetheless wrong. A row-count check
catches the rows that are **absent**, which no check on the contents of the present rows can
ever notice. Cost discipline decides the form: a suite that costs one full scan per rule will be
switched off within a month by whoever pays for it, so the rules are packed into one
aggregation.

In [10]:
def suite(dataframe):
    """Every metric the rules need, in one aggregation: one scan for the whole suite."""
    return dataframe.agg(
        F.count("*").alias("rows"),
        F.count(F.when(F.col("fare_amount").isNull(), 1)).alias("null_fares"),
        F.count(F.when(F.col("fare_amount") < 0, 1)).alias("negative_fares"),
        F.count(F.when(F.col("trip_time") <= 60, 1)).alias("short_trips"),
        F.count(F.when(F.col("total_amount") < F.col("fare_amount"), 1)).alias("total_below_fare"),
        F.count_distinct("medallion").alias("vehicles")).first()

# The rules, each with a declared consequence. "hard" stops the pipeline; "soft" records a
# number and continues until a threshold turns it hard.
def judge(metrics, expected_rows):
    rules = [
        ("row count within 20% of expectation", "hard",
         abs(metrics["rows"] - expected_rows) / expected_rows <= 0.20),
        ("no negative fares",                   "hard", metrics["negative_fares"] == 0),
        ("no total_amount below fare_amount",   "hard", metrics["total_below_fare"] == 0),
        ("null fares under 0.5%",               "soft",
         metrics["null_fares"] / metrics["rows"] < 0.005),
        ("sub-minute trips under 5%",           "soft",
         metrics["short_trips"] / metrics["rows"] < 0.05),
        ("at least 5,000 vehicles",             "hard", metrics["vehicles"] >= 5_000),
    ]
    failures = [(rule, kind) for rule, kind, passed in rules if not passed]
    for rule, kind, passed in rules:
        print(f"    {'pass' if passed else 'FAIL':4s}  [{kind}]  {rule}")
    return failures

baseline = good.count()
for label, batch in [("the good batch", good), ("the bad batch", bad)]:
    started = time.time()
    metrics = suite(batch)
    print(f"\n  {label}  ({time.time() - started:.1f}s for the whole suite)")
    print(f"    metrics: {dict(metrics.asDict())}")
    failures = judge(metrics, expected_rows=baseline)
    hard = [rule for rule, kind in failures if kind == "hard"]
    if hard:
        print(f"    -> STOP: {len(hard)} hard rule(s) failed: {hard[0]}")
    elif failures:
        print(f"    -> continue, with {len(failures)} soft failure(s) recorded")
    else:
        print("    -> publish")


  the good batch  (0.8s for the whole suite)
    metrics: {'rows': 1999999, 'null_fares': 0, 'negative_fares': 0, 'short_trips': 27071, 'total_below_fare': 0, 'vehicles': 10867}
    pass  [hard]  row count within 20% of expectation
    pass  [hard]  no negative fares
    pass  [hard]  no total_amount below fare_amount
    pass  [soft]  null fares under 0.5%
    pass  [soft]  sub-minute trips under 5%
    pass  [hard]  at least 5,000 vehicles
    -> publish



  the bad batch  (0.4s for the whole suite)
    metrics: {'rows': 998885, 'null_fares': 20081, 'negative_fares': 20253, 'short_trips': 13510, 'total_below_fare': 19801, 'vehicles': 9496}
    FAIL  [hard]  row count within 20% of expectation
    FAIL  [hard]  no negative fares
    FAIL  [hard]  no total_amount below fare_amount
    FAIL  [soft]  null fares under 0.5%
    pass  [soft]  sub-minute trips under 5%
    pass  [hard]  at least 5,000 vehicles
    -> STOP: 3 hard rule(s) failed: row count within 20% of expectation


The division of labour in that listing is the design: **measurement happens in the cluster**,
inside one `agg`, and **judgment happens in the driver**, over six plain numbers. Adding a rule
to `judge` costs nothing at all; adding a metric to `suite` costs nothing extra either, because
it rides the same scan.

Notice which check caught the column shift. Not the schema check, which passed. Not the null or
row-count checks, which the shift does not touch. The rule that caught it is
`total_amount < fare_amount` — a **range check between two columns**, which is the answer to
Exercise 7(c). A shift leaves every type intact, so only a statement about what the values must
*mean relative to each other* can see it.

### Reclaiming even that one scan: `Observation`

When the DataFrame is about to be written anyway, the audit can ride the write.

In [11]:
obs = Observation("publish")

audited = good.observe(
    obs,
    F.count(F.lit(1)).alias("rows"),
    F.count(F.when(F.col("fare_amount").isNull(), 1)).alias("null_fares"),
    F.count(F.when(F.col("fare_amount") < 0, 1)).alias("negative_fares"))

OUT = os.path.join(SCRATCH, "ch04-published")
started = time.time()
audited.write.mode("overwrite").parquet(OUT)
write_seconds = time.time() - started

print(f"write took {write_seconds:.1f}s")
print("metrics collected during that write:", obs.get)
print("\nno second pass over the data: the write itself paid for the audit.")

write took 0.7s
metrics collected during that write: {'rows': 1999999, 'null_fares': 0, 'negative_fares': 0}

no second pass over the data: the write itself paid for the audit.


In [12]:
# The guarantees, and the one limitation.
print("Observation guarantees:")
print("  * metrics are collected while the FIRST action executes (this write)")
print("  * subsequent actions cannot modify them -- obs.get is stable once set")
print("  * retrieval blocks until that action completes")
print()
before = dict(obs.get)
audited.write.mode("overwrite").parquet(OUT + "-again")   # a second action
print("obs.get unchanged after a second action:", dict(obs.get) == before)
print()
print("the limitation: Observation does not support streaming datasets, so the technique")
print("applies to batch writes only.")

Observation guarantees:
  * metrics are collected while the FIRST action executes (this write)
  * subsequent actions cannot modify them -- obs.get is stable once set
  * retrieval blocks until that action completes



obs.get unchanged after a second action: True

the limitation: Observation does not support streaming datasets, so the technique
applies to batch writes only.


## Where the checks go, and what failure means

Three placements repay their cost, and each check needs a declared consequence.

| Placement | What it protects against | Typical consequence |
|-----------|--------------------------|---------------------|
| immediately after ingestion | a bad batch consuming a whole pipeline's compute | hard for schema drift, soft with a threshold for bad-record rates |
| after the principal transformations | a bug in this chapter's own logic propagating | hard for invariants the transformation must preserve, such as a row count that a join should not have changed |
| immediately before publication | anything downstream inheriting a defect | hard for everything that corrupts consumers |

The one policy that is never correct is computing a check and examining it nowhere: tolerance
without a paper trail, the same lesson `try_cast` taught in
[4.2](04.02%20Cleaning%20the%20Taxi%20Data.ipynb) at the level of a single expression.

In [13]:
# The post-transformation placement, made concrete: a join must not change the row count.
vehicles = (df.select("medallion").distinct()
            .withColumn("make", F.lit("Ford")))
before_join = df.count()
after_join = df.join(vehicles, "medallion", "left").count()
print(f"rows before the enrichment join: {before_join:,}")
print(f"rows after                     : {after_join:,}")
assert after_join == before_join, "the join multiplied rows"
print("invariant held: a left join against a deduplicated reference table adds no rows.")

df.unpersist()
print("\nscratch:", SCRATCH)

rows before the enrichment join: 1,999,999
rows after                     : 1,999,999
invariant held: a left join against a deduplicated reference table adds no rows.

scratch: /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777


## Conclusion

* **Profile before believing.** `summary()` on one column showed a maximum two orders of
  magnitude above the median, which is a fact about the data no schema records.
* **The whole profile is one pass.** Nine metrics over four columns in a single `agg`, with the
  expressions built by a driver-side comprehension.
* **Cardinality against the row count has three readings**, and the useful one here was the
  column pair that *should* have been a key and was not.
* **Three kinds of check, and none substitutes for another.** The schema check is free and
  caught a rename and a type change while passing the bad batch entirely; the range checks
  caught the values that were the right type and wrong; the row-count check caught the rows
  that were not there at all.
* **A between-column range check is what catches a column shift**, the defect that
  [4.1](04.01%20Reader%20Policy%20and%20Corrupt%20Records.ipynb) showed no parse mode can see.
* **One scan for the whole suite**, or none at all with `Observation` on a write whose metrics
  are collected during the first action and cannot be changed by later ones.
* **Every rule carries a consequence**, hard or soft, decided in advance. A check computed and
  examined nowhere is the one policy that is never correct.